# Data Preprocessing Pipeline

This notebook handles the complete data preprocessing pipeline for the Telco Customer Churn dataset.

In [ ]:
# Install required packages
%pip install pandas scikit-learn pyarrow

# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from pathlib import Path

In [ ]:
# Configuration
DATA_PATH = "../data/customer_churn_dataset-training-master.csv"
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

In [ ]:
# Load dataset
print("Loading dataset...")
data = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {data.shape}")
print(f"Columns: {list(data.columns)}")
data.head()

In [ ]:
# Data inspection
print("=== Data Inspection ===")
print(f"\nShape: {data.shape}")
print(f"\nData types:\n{data.dtypes}")
print(f"\nMissing values:\n{data.isnull().sum()}")
print(f"\nStatistical summary:")
print(data.describe())

In [ ]:
# Handle missing values
print("=== Handling Missing Values ===")
print(f"Rows with missing values: {data.isnull().any(axis=1).sum()}")

# Remove rows with any missing values
data_clean = data.dropna()
print(f"Dataset shape after cleaning: {data_clean.shape}")
print(f"Removed {data.shape[0] - data_clean.shape[0]} rows")

In [ ]:
# Define feature categories
numerical_columns = ['Age', 'Tenure', 'Usage Frequency', 'Support Calls',
                     'Payment Delay', 'Total Spend', 'Last Interaction']

categorical_columns = ['Subscription Type', 'Contract Length']

identifier_column = 'CustomerID'
target_column = 'Churn'

print(f"Numerical features: {len(numerical_columns)}")
print(f"Categorical features: {len(categorical_columns)}")
print(f"Identifier: {identifier_column}")
print(f"Target: {target_column}")

In [ ]:
# Separate features and target
print("=== Separating Features and Target ===")

features = data_clean.drop(columns=[target_column])
labels = data_clean[target_column]

print(f"Features shape: {features.shape}")
print(f"Labels shape: {labels.shape}")

In [ ]:
# Train/test split
print("=== Train/Test Split ===")

X_train, X_test, y_train, y_test = train_test_split(
    features, labels,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"Training labels: {y_train.shape}")
print(f"Test labels: {y_test.shape}")

In [ ]:
# Scale numerical features
print("=== Scaling Numerical Features ===")

scaler = StandardScaler()

# Fit on training data
X_train_numerical = X_train[numerical_columns]
scaler.fit(X_train_numerical)

# Transform training data
X_train_scaled = scaler.transform(X_train_numerical)
X_train_scaled_df = pd.DataFrame(
    data=X_train_scaled,
    index=X_train.index,
    columns=numerical_columns
)

# Transform test data
X_test_numerical = X_test[numerical_columns]
X_test_scaled = scaler.transform(X_test_numerical)
X_test_scaled_df = pd.DataFrame(
    data=X_test_scaled,
    index=X_test.index,
    columns=numerical_columns
)

print(f"Scaled training data shape: {X_train_scaled_df.shape}")
print(f"Scaled test data shape: {X_test_scaled_df.shape}")

In [ ]:
# Encode categorical features
print("=== Encoding Categorical Features ===")

encoder = OneHotEncoder(sparse_output=False)

# Fit on training data
X_train_categorical = X_train[categorical_columns]
encoder.fit(X_train_categorical)

# Transform training data
X_train_encoded = encoder.transform(X_train_categorical)
X_train_encoded_df = pd.DataFrame(
    data=X_train_encoded,
    index=X_train.index,
    columns=encoder.get_feature_names_out()
)

# Transform test data
X_test_categorical = X_test[categorical_columns]
X_test_encoded = encoder.transform(X_test_categorical)
X_test_encoded_df = pd.DataFrame(
    data=X_test_encoded,
    index=X_test.index,
    columns=encoder.get_feature_names_out()
)

print(f"Encoded training data shape: {X_train_encoded_df.shape}")
print(f"Encoded test data shape: {X_test_encoded_df.shape}")
print(f"Encoded columns: {list(encoder.get_feature_names_out())}")

In [ ]:
# Combine processed features
print("=== Combining Processed Features ===")

# Combine for training set
X_train_processed = pd.concat([
    X_train[identifier_column],
    X_train_scaled_df,
    X_train_encoded_df
], axis=1)

# Combine for test set
X_test_processed = pd.concat([
    X_test[identifier_column],
    X_test_scaled_df,
    X_test_encoded_df
], axis=1)

print(f"Processed training data shape: {X_train_processed.shape}")
print(f"Processed test data shape: {X_test_processed.shape}")

In [ ]:
# Save processed data
print("=== Saving Processed Data ===")

train_path = OUTPUT_DIR / "train.parquet"
test_path = OUTPUT_DIR / "test.parquet"
train_labels_path = OUTPUT_DIR / "train_labels.parquet"
test_labels_path = OUTPUT_DIR / "test_labels.parquet"

X_train_processed.to_parquet(train_path)
X_test_processed.to_parquet(test_path)
y_train.to_frame().to_parquet(train_labels_path)
y_test.to_frame().to_parquet(test_labels_path)

print(f"Saved training data: {train_path}")
print(f"Saved test data: {test_path}")
print(f"Saved training labels: {train_labels_path}")
print(f"Saved test labels: {test_labels_path}")

In [ ]:
# Save preprocessing objects
print("=== Saving Preprocessing Objects ===")

import pickle

scaler_path = OUTPUT_DIR / "scaler.pkl"
encoder_path = OUTPUT_DIR / "encoder.pkl"

with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)

with open(encoder_path, 'wb') as f:
    pickle.dump(encoder, f)

print(f"Saved scaler: {scaler_path}")
print(f"Saved encoder: {encoder_path}")

In [ ]:
# Summary
print("=== Preprocessing Summary ===")
print(f"\nOriginal dataset: {data.shape}")
print(f"After cleaning: {data_clean.shape}")
print(f"Training set: {X_train_processed.shape}")
print(f"Test set: {X_test_processed.shape}")
print(f"\nNumerical features scaled: {len(numerical_columns)}")
print(f"Categorical features encoded: {len(categorical_columns)}")
print(f"Total features after preprocessing: {X_train_processed.shape[1] - 1}")  # -1 for CustomerID
print(f"\nChurn rate in training: {y_train.mean():.2%}")
print(f"Churn rate in test: {y_test.mean():.2%}")